# Support-point CNN supervised training — Kaggle CUDA-BP notebook

This notebook is prepared for the `cnn-support-point-estimator` branch. It:

1. clones the current branch into `/kaggle/working`,
2. samples support-point configurations in the 7×7 lattice (`0..48`),
3. labels them with the **exact CUDA bit-parallel distance kernel** (`kernel_cuda_bp.py`),
4. trains the lightweight support-point CNN/ridge estimator (`support_point_cnn_estimator.py`), and
5. reports train/validation MSE/MAE/RMSE losses and **accuracy = percentage of predictions within ±3 of the actual minimum distance**.

Important separation:
- **Exact CUDA labeling** below is the supervised-data/certification step. It uses `DistanceOracleCUDABP.max_zeros_batch(..., sample_count=0)` and records `min_distance = 49 - max_zeros`.
- **CNN inference** is heuristic only. It is useful for ranking or triage, but it does **not** certify distances. Re-check selected supports with the CUDA kernel before treating a value as exact.

## Practical Kaggle setup notes

- In Kaggle, choose **Settings → Accelerator → GPU**. T4/P100/A100 are all fine; exact labeling is much faster on A100/T4 than CPU fallback workflows.
- Turn **Internet on** for the clone cell, or attach this repository as a Kaggle Dataset and replace the clone cell with a local copy command.
- The branch named below must be pushed to GitHub before running this notebook on Kaggle.
- Kaggle usually includes CuPy for NVIDIA GPUs. If the CUDA import cell fails, restart with a GPU accelerator and install the matching CuPy package only if necessary.
- Exact labeling is exponential in support size: exact CUDA-BP supports `k <= 21`, but practical supervised dataset generation should start around `k <= 7..9`. Increase `N_SAMPLES`, `K_MAX`, and `LABEL_BATCH_SIZE` gradually.
- Artifacts are written under `/kaggle/working/support_point_cnn_runs` so they can be downloaded from the Kaggle output panel.

In [ ]:
# Clone the current branch.
# If you forked the repo, update REPO_URL. If the branch name changed, update BRANCH.
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/t-ravnushkin/generalized-toric-gpu-search.git"
BRANCH = "cnn-support-point-estimator"
REPO_DIR = Path("/kaggle/working/generalized-toric-gpu-search")

if REPO_DIR.exists():
    print(f"Removing existing checkout: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

cmd = ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Checked out:")
subprocess.run(["git", "status", "--short", "--branch"], check=True)

In [ ]:
# CUDA and project imports. This notebook requires CUDA for exact labeling.
import importlib
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

import cupy as cp
n_gpu = cp.cuda.runtime.getDeviceCount()
if n_gpu <= 0:
    raise RuntimeError("No CUDA GPU found. In Kaggle, enable Settings → Accelerator → GPU and restart.")

print(f"CuPy {cp.__version__}; CUDA devices: {n_gpu}")
for i in range(n_gpu):
    props = cp.cuda.runtime.getDeviceProperties(i)
    name = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    print(f"  device {i}: {name}, {props.get('multiProcessorCount', '?')} SMs, {props['totalGlobalMem'] / 1024**3:.1f} GiB")

# The estimator has no GPU dependency; it is a NumPy fixed-CNN feature extractor + ridge regression.
from support_point_cnn_estimator import (
    GRID,
    N_POINTS,
    Example,
    RidgeCnnEstimator,
    one_point_extensions,
    train_estimator,
    train_validation_split,
)

# Avoid importing precompute.py here: it imports pyopencl at module import time. For this CUDA-only notebook,
# build the same evaluation matrix directly from gf8.py.
from gf8 import gf8_mul, gf8_pow
from kernel_cuda_bp import init_cuda_oracles_bp

print(f"Grid={GRID}x{GRID}, N_POINTS={N_POINTS}")

In [ ]:
# Configuration. Keep K_MAX modest until you have timed your Kaggle GPU.
SEED = 20260608
N_SAMPLES = 256
K_MIN = 2
K_MAX = 7              # exact labels are exponential in k; start small, then increase
LABEL_BATCH_SIZE = 256 # supports per CUDA launch group, grouped by k
VAL_FRACTION = 0.20
RIDGE = 1e-2
ACCURACY_TOLERANCE = 3.0

RUN_DIR = Path("/kaggle/working/support_point_cnn_runs")
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATASET_JSONL = RUN_DIR / "support_point_cuda_bp_labeled.jsonl"
MODEL_NPZ = RUN_DIR / "support_point_cnn_model.npz"
PREDICTIONS_CSV = RUN_DIR / "support_point_cnn_predictions.csv"

print({
    "N_SAMPLES": N_SAMPLES,
    "K_MIN": K_MIN,
    "K_MAX": K_MAX,
    "LABEL_BATCH_SIZE": LABEL_BATCH_SIZE,
    "RUN_DIR": str(RUN_DIR),
})

In [ ]:
# Support sampling utilities. These only choose candidate supports; no labels are created here.
def sample_supports(n: int, k_min: int, k_max: int, seed: int) -> list[tuple[int, ...]]:
    if n <= 0:
        raise ValueError("n must be positive")
    if not (1 <= k_min <= k_max <= N_POINTS):
        raise ValueError(f"expected 1 <= k_min <= k_max <= {N_POINTS}")
    rng = random.Random(seed)
    supports: list[tuple[int, ...]] = []
    seen: set[tuple[int, ...]] = set()
    ks = list(range(k_min, k_max + 1))
    attempts = 0
    while len(supports) < n:
        attempts += 1
        if attempts > max(1000, n * 500):
            raise RuntimeError("could not generate enough unique supports; lower n or widen k range")
        k = ks[len(supports) % len(ks)]  # roughly stratified by k
        support = tuple(sorted(rng.sample(range(N_POINTS), k)))
        if support in seen:
            continue
        seen.add(support)
        supports.append(support)
    random.Random(seed + 1).shuffle(supports)
    return supports

def build_eval_matrix_cuda_input() -> np.ndarray:
    lattice = [(a, b) for a in range(GRID) for b in range(GRID)]
    torus = [(x, y) for x in range(1, 8) for y in range(1, 8)]
    M = np.zeros((N_POINTS, N_POINTS), dtype=np.int32)
    for i, (a, b) in enumerate(lattice):
        for j, (x, y) in enumerate(torus):
            M[i, j] = gf8_mul(gf8_pow(x, a), gf8_pow(y, b))
    return M

supports = sample_supports(N_SAMPLES, K_MIN, K_MAX, SEED)
print(f"Sampled {len(supports)} supports")
print(pd.Series([len(s) for s in supports]).value_counts().sort_index().rename("count_by_k"))
print("First 5 supports:", supports[:5])

## Exact CUDA-BP labeling step

This is the only step that creates ground-truth labels. It uses the repository CUDA bit-parallel kernel with `sample_count=0` and `target_distance=1` so each supported coefficient space is fully scanned for `k <= 21`.

The CNN is not involved in this cell.

In [ ]:
def label_supports_cuda_bp(
    supports: list[tuple[int, ...]],
    *,
    label_batch_size: int = 256,
    dataset_path: Path | None = None,
) -> list[dict]:
    if any(len(s) > 21 for s in supports):
        too_large = sorted({len(s) for s in supports if len(s) > 21})
        raise ValueError(f"Exact CUDA-BP labeling supports k <= 21; got k values {too_large}")

    M = build_eval_matrix_cuda_input()
    oracles, sm_count = init_cuda_oracles_bp(M)
    oracle = oracles[0]  # Kaggle usually has one GPU; the batch kernel already parallelizes across supports.
    print(f"CUDA-BP exact labeler ready on device 0; sm_count={sm_count}")

    grouped: dict[int, list[tuple[int, tuple[int, ...]]]] = defaultdict(list)
    for pos, support in enumerate(supports):
        grouped[len(support)].append((pos, support))

    rows_by_pos: dict[int, dict] = {}
    t0 = time.perf_counter()
    for k, items in sorted(grouped.items()):
        print(f"Labeling k={k}: {len(items)} supports")
        for start in range(0, len(items), label_batch_size):
            chunk = items[start:start + label_batch_size]
            batch = [list(support) for _, support in chunk]
            # target_distance=1 => tmax_zeros=48. Except for degenerate distance-0 supports,
            # this forces a full exact scan; sample_count=0 disables sampling.
            max_zeros = oracle.max_zeros_batch(batch, target_distance=1, sample_count=0)
            for (pos, support), mz in zip(chunk, max_zeros, strict=True):
                mz = int(mz)
                rows_by_pos[pos] = {
                    "indices": list(support),
                    "k": len(support),
                    "max_zeros": mz,
                    "min_distance": int(N_POINTS - mz),
                    "label_backend": "cuda-bp-exact",
                    "sample_count": 0,
                }
        cp.cuda.Device(0).synchronize()
        elapsed = time.perf_counter() - t0
        print(f"  done k={k}; elapsed={elapsed:.1f}s")

    rows = [rows_by_pos[i] for i in range(len(supports))]
    if dataset_path is not None:
        with dataset_path.open("w") as f:
            for row in rows:
                f.write(json.dumps(row) + "\n")
        print(f"Wrote exact labeled dataset: {dataset_path}")
    return rows

labeled_rows = label_supports_cuda_bp(supports, label_batch_size=LABEL_BATCH_SIZE, dataset_path=DATASET_JSONL)
labels_df = pd.DataFrame(labeled_rows)
print(labels_df.groupby("k")["min_distance"].agg(["count", "min", "mean", "max"]))
labels_df.head()

## Heuristic CNN training/inference step

From here on, the exact labels are treated as a supervised dataset. The estimator is a heuristic NumPy model: fixed 3×3 convolutional features over the 7×7 support mask followed by ridge regression.

The losses and ±3 accuracy below measure how well the heuristic reproduces CUDA labels on this sampled dataset; they are not distance certificates.

In [ ]:
def examples_from_rows(rows: list[dict]) -> list[Example]:
    return [Example(tuple(int(i) for i in row["indices"]), float(row["min_distance"])) for row in rows]

def regression_metrics(estimator: RidgeCnnEstimator, examples: list[Example], tolerance: float = 3.0) -> dict:
    if not examples:
        return {"count": 0, "mae": np.nan, "rmse": np.nan, "accuracy_within_pm3": np.nan}
    y = np.asarray([ex.target for ex in examples], dtype=np.float64)
    pred = estimator.predict_indices([ex.indices for ex in examples])
    err = pred - y
    mse = float(np.mean(err * err))
    return {
        "count": len(examples),
        "mse_loss": mse,
        "mae_loss": float(np.mean(np.abs(err))),
        "rmse_loss": float(np.sqrt(mse)),
        "bias": float(np.mean(err)),
        "accuracy_within_pm3": float(np.mean(np.abs(err) <= tolerance) * 100.0),
    }

examples = examples_from_rows(labeled_rows)
train_examples, val_examples = train_validation_split(examples, VAL_FRACTION, SEED)

estimator = train_estimator(train_examples, ridge=RIDGE)
estimator.save(MODEL_NPZ)

metrics_df = pd.DataFrame([
    {"split": "train", **regression_metrics(estimator, train_examples, ACCURACY_TOLERANCE)},
    {"split": "validation", **regression_metrics(estimator, val_examples, ACCURACY_TOLERANCE)},
])
print(f"Saved model: {MODEL_NPZ}")
metrics_df

In [ ]:
# Per-example predictions for inspection/download.
def prediction_frame(split_name: str, examples: list[Example]) -> pd.DataFrame:
    pred = estimator.predict_indices([ex.indices for ex in examples])
    actual = np.asarray([ex.target for ex in examples], dtype=np.float64)
    return pd.DataFrame({
        "split": split_name,
        "indices": [list(ex.indices) for ex in examples],
        "k": [len(ex.indices) for ex in examples],
        "actual_min_distance_cuda_bp": actual.astype(int),
        "predicted_min_distance_cnn": pred,
        "error_pred_minus_actual": pred - actual,
        "within_pm3": np.abs(pred - actual) <= ACCURACY_TOLERANCE,
    })

pred_df = pd.concat([
    prediction_frame("train", train_examples),
    prediction_frame("validation", val_examples),
], ignore_index=True)
pred_df.to_csv(PREDICTIONS_CSV, index=False)
print(f"Wrote predictions: {PREDICTIONS_CSV}")
print(pred_df.groupby(["split", "k"])["within_pm3"].agg(count="count", accuracy_pct=lambda s: 100 * s.mean()).round(2))
pred_df.head(10)

In [ ]:
# Optional diagnostic plot.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
colors = {"train": "tab:blue", "validation": "tab:orange"}
for split, part in pred_df.groupby("split"):
    ax.scatter(
        part["actual_min_distance_cuda_bp"],
        part["predicted_min_distance_cnn"],
        label=split,
        alpha=0.75,
        s=28,
        color=colors.get(split),
    )
lo = min(pred_df["actual_min_distance_cuda_bp"].min(), pred_df["predicted_min_distance_cnn"].min()) - 1
hi = max(pred_df["actual_min_distance_cuda_bp"].max(), pred_df["predicted_min_distance_cnn"].max()) + 1
ax.plot([lo, hi], [lo, hi], "k--", linewidth=1, label="perfect")
ax.fill_between([lo, hi], [lo - 3, hi - 3], [lo + 3, hi + 3], color="gray", alpha=0.15, label="±3")
ax.set_xlabel("Actual min distance (CUDA-BP exact label)")
ax.set_ylabel("Predicted min distance (CNN heuristic)")
ax.set_title("Support-point CNN predictions vs exact CUDA labels")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## Heuristic-only ranking example

This cell uses only the trained CNN estimator to rank candidate one-point extensions. These numbers are **predictions**, not exact labels. Use the optional re-check cell after this if you want exact CUDA distances for selected candidates.

In [ ]:
# Choose a base support and rank all one-point extensions by predicted min distance.
BASE_SUPPORT = (0, 1, 7)
TOP_N = 10

candidates = one_point_extensions(BASE_SUPPORT)
scores = estimator.predict_indices(candidates)
order = np.argsort(scores)[::-1][:TOP_N]
ranked = pd.DataFrame([
    {
        "rank": rank,
        "added_point": sorted(set(candidates[int(pos)]) - set(BASE_SUPPORT))[0],
        "support": list(candidates[int(pos)]),
        "predicted_min_distance_cnn": float(scores[int(pos)]),
        "inference_type": "cnn-heuristic-not-certified",
    }
    for rank, pos in enumerate(order, start=1)
])
ranked

## Optional exact CUDA re-check of heuristic top candidates

Run this only when you want to convert heuristic rankings into exact labels for the selected candidates. This is a separate exact-labeling call and can be appended to your supervised dataset.

In [ ]:
# Exact CUDA labels for the heuristic top-N candidates. Keep this separate from CNN inference.
RECHECK_TOP_CANDIDATES = True

if RECHECK_TOP_CANDIDATES:
    top_supports = [tuple(row) for row in ranked["support"]]
    recheck_rows = label_supports_cuda_bp(top_supports, label_batch_size=LABEL_BATCH_SIZE, dataset_path=None)
    recheck_df = pd.DataFrame(recheck_rows).rename(columns={"min_distance": "actual_min_distance_cuda_bp"})
    ranked_for_merge = ranked.copy()
    ranked_for_merge["support_key"] = ranked_for_merge["support"].apply(tuple)
    recheck_df["support_key"] = recheck_df["indices"].apply(tuple)
    merged = ranked_for_merge.merge(
        recheck_df[["support_key", "actual_min_distance_cuda_bp", "max_zeros", "label_backend"]],
        on="support_key",
        how="left",
    ).drop(columns=["support_key"])
    merged["error_pred_minus_actual"] = merged["predicted_min_distance_cnn"] - merged["actual_min_distance_cuda_bp"]
    display(merged)
else:
    print("Skipped exact CUDA re-check.")

## Outputs

Download these from Kaggle's output panel after the run:

- `support_point_cuda_bp_labeled.jsonl` — supervised dataset with exact CUDA-BP labels.
- `support_point_cnn_model.npz` — saved heuristic CNN/ridge estimator.
- `support_point_cnn_predictions.csv` — train/validation predictions, errors, and ±3 flags.

Remember: only rows labeled by `label_backend = cuda-bp-exact` are exact distance evaluations. CNN predictions are for ranking and triage only.